# muon_stalactite：反冲核产生位置的 z 密度分布
依据 `Data/Geant4_data/src/SteppingAction.cc` 和 `run_geant.ipynb`。需要 NumPy 和 PyROOT（在安装 ROOT 的 Python/Jupyter 环境运行）。

**物理口径**：当前输出是初级粒子 TrackID=1 在 Target 内直接产生、动能 >1 keV、质量 >900 MeV 的次级粒子；本程序进一步选取核素名称及轻核，排除自由质子和中子。这不是完整级联反冲产额，也不保证每个核都是靶核弹性反冲。

旧数据只有 z，没有 x/y。三维图的三个轴为 z、入射能量、dN/dz，而非真实 xyz 空间分布。不能将这些数据当作钟乳石形状或横向均匀分布；真实空间 TH3 图需重新记录 x/y/z 并运行模拟。


In [1]:
from pathlib import Path
import json, re, csv
from array import array
import numpy as np

PROJECT = next((p for p in [Path.cwd(), Path.cwd()/"primus-paleo", *Path.cwd().parents]
                if (p/"Data/Geant4_data").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("请从 primus-paleo 目录启动 notebook")
MATERIAL = "Olivine"
PARTICLE = "mu-"
DATA = PROJECT / "Data/Geant4_data" / MATERIAL / PARTICLE
N_Z = 100
# 不从记录数推断入射粒子数。若有可靠运行日志，可填每个文件的 beamOn 数。
N_PRIMARY_PER_FILE = None
# 无 geometry.json 时必须手动指定真实几何，不能由样本极值推断。
GEOMETRY_OVERRIDE = None  # 例如 {"target_length_m": 200, "target_radius_m": 10}
OUT = PROJECT / "output" / "muon_stalactite" / MATERIAL / PARTICLE
OUT.mkdir(parents=True, exist_ok=True)


## 读取原始数据
无表头六列依次为：粒子名、质量 MeV、次级动能 MeV、z mm、初级步前/步后动能 MeV。文件名中的能量单位 GeV。每个文件单独统计，避免混合不同能量的模拟曝光。输出为线密度；圆柱横截面平均体积密度另行输出。

In [2]:
geometry = GEOMETRY_OVERRIDE
if geometry is None:
    geometry = json.loads((DATA / "geometry.json").read_text())
length, radius = float(geometry["target_length_m"]), float(geometry["target_radius_m"])
assert np.isfinite([length, radius]).all() and length > 0 and radius > 0
assert N_Z > 0
if N_PRIMARY_PER_FILE is not None:
    assert np.isfinite(N_PRIMARY_PER_FILE) and N_PRIMARY_PER_FILE > 0
files = sorted(DATA.glob("outNuclei_*.txt"), key=lambda p: float(p.stem.split("_")[-1]))
if not files:
    raise FileNotFoundError(f"没有输入文件：{DATA}")
energies = np.array([float(p.stem.split("_")[-1]) for p in files])
assert np.all(energies > 0) and np.all(np.diff(energies) > 0)
edges = np.linspace(-length/2, length/2, N_Z+1)
centres = (edges[:-1] + edges[1:])/2
counts = np.zeros((len(files), N_Z), dtype=np.int64)
summary = []
def is_nucleus(name):
    return name in {"alpha", "deuteron", "triton", "He3"} or re.fullmatch(r"[A-Z][a-z]?\d+(?:\[.*\])?", name) is not None
assert is_nucleus("O16") and is_nucleus("Mg24[1.0]")
assert not is_nucleus("neutron") and not is_nucleus("proton")
for i, path in enumerate(files):
    selected = outside = rows = 0
    positions = []
    with path.open() as stream:
        for line_no, line in enumerate(stream, 1):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 6:
                raise ValueError(f"{path}:{line_no}: 预期六列，实际 {len(parts)}")
            values = np.array(parts[1:], dtype=float)
            if not np.isfinite(values).all():
                raise ValueError(f"{path}:{line_no}: 非有限数值")
            rows += 1
            if not is_nucleus(parts[0]) or values[1] <= 0.001:
                continue
            selected += 1
            z = values[2] / 1000  # mm -> m
            if z < edges[0] or z > edges[-1]:
                outside += 1
            else:
                positions.append(z)
    counts[i], _ = np.histogram(positions, bins=edges)
    assert counts[i].sum() + outside == selected
    summary.append(dict(file=path.name, rows=rows, nuclei=selected, outside=outside))
if any(s["outside"] for s in summary):
    raise ValueError("存在几何范围外的核，请核对 geometry.json 和数据来源")
normalization = 1.0 if N_PRIMARY_PER_FILE is None else float(N_PRIMARY_PER_FILE)
density = counts / np.diff(edges)[None, :] / normalization
error = np.sqrt(counts) / np.diff(edges)[None, :] / normalization
volume_density = density / (np.pi * radius**2)
assert np.allclose((density*np.diff(edges)).sum(axis=1), counts.sum(axis=1)/normalization)
print(f"读取 {len(files)} 个能量点，选中 {counts.sum()} 个核；空文件也保留为零计数。")
print("归一化：", "原始计数/m" if N_PRIMARY_PER_FILE is None else "核/(入射粒子 m)")


读取 90 个能量点，选中 1107297 个核；空文件也保留为零计数。
归一化： 原始计数/m


In [3]:
with (OUT / "z_density.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["energy_GeV", "z_m", "count", "dN_dz", "poisson_error", "cross_section_mean_dN_dV"])
    for i, energy in enumerate(energies):
        writer.writerows(zip(np.repeat(energy, N_Z), centres, counts[i], density[i], error[i], volume_density[i]))
(OUT / "provenance.json").write_text(json.dumps(dict(material=MATERIAL, particle=PARTICLE,
    geometry=geometry, n_primary_per_file=N_PRIMARY_PER_FILE, files=summary), indent=2), encoding="utf-8")


10633

## ROOT 三维曲面和 z 分布
曲面高度是每个能量点的 dN/dz；没有除以能量 bin 宽度，因此不是 d²N/(dz dE)。默认绘制各能量的原始模拟计数密度，不代表宇宙线通量加权结果。误差为计数的 sqrt(N) 描述性误差，同一次初级事件内关联未包含。

In [ ]:
try:
    import ROOT
except ImportError as exc:
    raise ImportError("统计和 CSV 已生成；绘图需要切换到已安装 PyROOT 的 Jupyter 内核。") from exc
ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
# 柔和的顺序色标：浅蓝低值、深蓝高值；不平滑原始统计。
stops = array('d', [0., 0.25, 0.5, 0.75, 1.])
red = array('d', [0.96, 0.76, 0.42, 0.18, 0.04])
green = array('d', [0.98, 0.86, 0.68, 0.46, 0.22])
blue = array('d', [1., 0.94, 0.84, 0.70, 0.44])
ROOT.TColor.CreateGradientColorTable(5, stops, red, green, blue, 255)
ROOT.gStyle.SetNumberContours(255)
ROOT.gStyle.SetCanvasColor(0)
ROOT.gStyle.SetPadColor(0)
ROOT.gStyle.SetTitleBorderSize(0)
ROOT.gStyle.SetTextFont(42)
ROOT.gStyle.SetLabelFont(42, "XYZ")
ROOT.gStyle.SetTitleFont(42, "XYZ")
ROOT.gStyle.SetLabelSize(0.032, "XYZ")
ROOT.gStyle.SetTitleSize(0.038, "XYZ")
log_e = np.log10(energies)
if len(log_e) == 1:
    e_edges = np.array([log_e[0]-0.5, log_e[0]+0.5])
else:
    e_edges = np.r_[log_e[0]-(log_e[1]-log_e[0])/2, (log_e[1:]+log_e[:-1])/2,
                    log_e[-1]+(log_e[-1]-log_e[-2])/2]
unit = "N / m" if N_PRIMARY_PER_FILE is None else "N / primary / m"
h = ROOT.TH2D("z_energy_density", f"{MATERIAL} {PARTICLE};z [m];log10(E / GeV);{unit}",
              N_Z, array('d', edges), len(energies), array('d', e_edges))
h.SetDirectory(0)
for i in range(len(energies)):
    for j in range(N_Z):
        h.SetBinContent(j+1, i+1, float(density[i,j]))
        h.SetBinError(j+1, i+1, float(error[i,j]))
c3 = ROOT.TCanvas("density3d", "Recoil production density", 1100, 800)
c3.SetLeftMargin(0.14)
c3.SetRightMargin(0.16)
c3.SetBottomMargin(0.16)
c3.SetTheta(28)
c3.SetPhi(40)
h.GetXaxis().SetTitleOffset(1.7)
h.GetYaxis().SetTitleOffset(1.9)
h.GetZaxis().SetTitleOffset(1.5)
h.Draw("SURF2")
c3.SaveAs(str(OUT / "density_3d.png"))
# 选择计数最多的能量点展示一维分布；ROOT 文件保存所有能量点。
best = int(np.argmax(counts.sum(axis=1)))
profiles = []
for i, energy in enumerate(energies):
    profile = ROOT.TH1D(f"z_profile_{i}", f"E={energy:g} GeV;z [m];{unit}", N_Z, array('d', edges))
    profile.SetDirectory(0)
    for j in range(N_Z):
        profile.SetBinContent(j+1, float(density[i,j]))
        profile.SetBinError(j+1, float(error[i,j]))
    profiles.append(profile)
c1 = ROOT.TCanvas("density_z", "Z density", 1000, 650)
profiles[best].Draw("E1")
c1.SaveAs(str(OUT / "density_z.png"))
fout = ROOT.TFile(str(OUT / "muon_stalactite.root"), "RECREATE")
if fout.IsZombie():
    raise OSError("ROOT 输出文件创建失败")
h.Write()
for profile in profiles:
    profile.Write()
c3.Write()
c1.Write()
fout.Close()
from IPython.display import display, Image
display(Image(filename=str(OUT / "density_3d.png")))
display(Image(filename=str(OUT / "density_z.png")))


## 补充图形：热图、等高线和能量切片
热图适合查看全局分布；对数色标用于比较跨数量级的非零密度（零计数不代表正值，不添加伪计数）。等高线展示相同密度的位置；能量切片便于定量比较。所有图使用原始 bin，不做平滑。不同能量曝光不一致时，原始计数不能直接解释为产额差异。

下面的 `SLICE_ENERGIES_GEV` 可改为关注的入射能量，自动选择数据中最近的能量点。图片同时输出 PNG/PDF，图对象写入独立 ROOT 文件。

In [ ]:
SLICE_ENERGIES_GEV = [0.01, 0.1, 1., 10., 100.]
extra_canvases = []
extra_histograms = []
def save_extra(canvas, name):
    canvas.Modified()
    canvas.Update()
    for ext in ("png", "pdf"):
        canvas.SaveAs(str(OUT / f"{name}.{ext}"))
    extra_canvases.append(canvas)
    display(Image(filename=str(OUT / f"{name}.png")))

for name, option, log_scale in [
    ("density_heatmap", "COLZ", False),
    ("density_heatmap_log", "COLZ", True),
    ("density_contours", "CONT1Z", False),
]:
    positive = density[density > 0]
    if log_scale and positive.size == 0:
        print("没有正密度，跳过对数热图")
        continue
    canvas = ROOT.TCanvas(name, name, 1100, 760)
    canvas.SetLeftMargin(0.12)
    canvas.SetRightMargin(0.19)
    canvas.SetBottomMargin(0.13)
    plot = h.Clone(name + "_hist")
    plot.SetDirectory(0)
    plot.GetXaxis().SetTitleOffset(1.2)
    plot.GetYaxis().SetTitleOffset(1.2)
    plot.GetZaxis().SetTitleOffset(1.3)
    if log_scale:
        canvas.SetLogz(True)
        plot.SetMinimum(float(positive.min()) * 0.9)
        plot.SetMaximum(float(positive.max()) * 1.1)
    else:
        plot.SetMinimum(0.)
    if option == "CONT1Z":
        plot.SetContour(12)
    plot.Draw(option)
    extra_histograms.append(plot)
    save_extra(canvas, name)

requested = np.asarray(SLICE_ENERGIES_GEV, dtype=float)
if requested.size == 0 or not np.isfinite(requested).all() or np.any(requested <= 0):
    raise ValueError("切片能量必须为有限正数，且至少选一个")
indices = sorted({int(np.argmin(abs(np.log(energies / e)))) for e in requested})
canvas = ROOT.TCanvas("energy_slices", "Energy slices", 1100, 760)
canvas.SetLeftMargin(0.13)
canvas.SetBottomMargin(0.13)
canvas.SetRightMargin(0.05)
legend = ROOT.TLegend(0.15, 0.68, 0.45, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
colors = ["#2864A0", "#D17832", "#329681", "#9865A8", "#C55368"]
ymax = max(float(np.max(density[i] + error[i])) for i in indices)
for k, i in enumerate(indices):
    curve = profiles[i].Clone(f"slice_{i}")
    curve.SetDirectory(0)
    curve.SetTitle(f"{MATERIAL} {PARTICLE}: energy slices;z [m];{unit}")
    curve.SetLineColor(ROOT.TColor.GetColor(colors[k % len(colors)]))
    curve.SetLineStyle(1 + k // len(colors))
    curve.SetLineWidth(2)
    curve.SetMinimum(0.)
    curve.SetMaximum(max(1., ymax * 1.25))
    curve.Draw("HIST" if k == 0 else "HIST SAME")
    legend.AddEntry(curve, f"{energies[i]:.4g} GeV", "l")
    extra_histograms.append(curve)
legend.Draw()
save_extra(canvas, "density_energy_slices")
extra_file = ROOT.TFile(str(OUT / "muon_stalactite_extra.root"), "RECREATE")
if extra_file.IsZombie():
    raise OSError("无法创建补充图 ROOT 文件")
for obj in extra_histograms + extra_canvases:
    obj.Write()
extra_file.Close()
